In [112]:
import numpy as np 
import pandas as pd
import seaborn as sns 
import matplotlib.pyplot as plt 
import seaborn as sns

In [113]:
data = pd.read_csv(r"C:\Users\91972\Downloads\Heart_Disease_Prediction.csv")
df = data.copy()
df.info()
df.sample(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270 entries, 0 to 269
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Age                      270 non-null    int64  
 1   Sex                      270 non-null    int64  
 2   Chest pain type          270 non-null    int64  
 3   BP                       270 non-null    int64  
 4   Cholesterol              270 non-null    int64  
 5   FBS over 120             270 non-null    int64  
 6   EKG results              270 non-null    int64  
 7   Max HR                   270 non-null    int64  
 8   Exercise angina          270 non-null    int64  
 9   ST depression            270 non-null    float64
 10  Slope of ST              270 non-null    int64  
 11  Number of vessels fluro  270 non-null    int64  
 12  Thallium                 270 non-null    int64  
 13  Heart Disease            270 non-null    object 
dtypes: float64(1), int64(12), 

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
61,51,0,4,130,305,0,0,142,1,1.2,2,0,7,Presence
76,45,1,4,104,208,0,2,148,1,3.0,2,0,3,Absence


In [114]:
df.describe()

,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium
count,270.000000,270.000000,270.000000,270.000000,270.000000,270.000000,270.000000,270.000000,270.000000,270.00000,270.000000,270.000000,270.000000
mean,54.433333,0.677778,3.174074,131.344444,249.659259,0.148148,1.022222,149.677778,0.329630,1.05000,1.585185,0.670370,4.696296
std,9.109067,0.468195,0.950090,17.861608,51.686237,0.355906,0.997891,23.165717,0.470952,1.14521,0.614390,0.943896,1.940659
min,29.000000,0.000000,1.000000,94.000000,126.000000,0.000000,0.000000,71.000000,0.000000,0.00000,1.000000,0.000000,3.000000
25%,48.000000,0.000000,3.000000,120.000000,213.000000,0.000000,0.000000,133.000000,0.000000,0.00000,1.000000,0.000000,3.000000
50%,55.000000,1.000000,3.000000,130.000000,245.000000,0.000000,2.000000,153.500000,0.000000,0.80000,2.000000,0.000000,3.000000
75%,61.000000,1.000000,4.000000,140.000000,280.000000,0.000000,2.000000,166.000000,1.000000,1.60000,2.000000,1.000000,7.000000
max,77.000000,1.000000,4.000000,200.000000,564.000000,1.000000,2.000000,202.000000,1.000000,6.20000,3.000000,3.000000,7.000000


## So this is a Heart Disease Prediction DataSet
# So we will apply Logistic Regression to it 
we will apply Logositc Regression in the following manner
1. Simple Logistic Regression 
    1. using Ridge
    2. Lasso 
    3. Eastic Net regularisation
2. Polynomial logistic regression
   
# We will check accuracy of each one with CV to find the better one  


In [115]:
from sklearn.model_selection import train_test_split

X_train,X_test ,y_train,y_test = train_test_split(df.drop('Heart Disease',axis=1),df.loc[:,['Heart Disease']],test_size=0.2 , random_state=909)

In [126]:

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler,PolynomialFeatures
from sklearn.metrics import confusion_matrix,accuracy_score,f1_score,recall_score,precision_score
from sklearn.compose import ColumnTransformer

X_train.iloc[:,[0,3,4,7]].describe()

,Age,BP,Cholesterol,Max HR
count,216.000000,216.000000,216.000000,216.000000
mean,54.708333,130.810185,250.939815,150.027778
std,9.150581,17.472798,53.629921,23.409185
min,29.000000,94.000000,126.000000,71.000000
25%,48.000000,120.000000,215.750000,135.500000
50%,56.000000,130.000000,245.000000,154.000000
75%,62.000000,140.000000,277.000000,167.250000
max,77.000000,200.000000,564.000000,202.000000


In [117]:
# Pipe line 1 , Simple Logistic Regression
sd=StandardScaler()
ct = ColumnTransformer(transformers=[('Stand',sd,[0,3,4,7])],remainder='passthrough')
pipe1 = Pipeline(steps=[('Std',ct),
                        ('Lg',LogisticRegression(penalty='elasticnet',max_iter=10000,solver='saga'))])

# Using Grid Search CV
from sklearn.model_selection import GridSearchCV

g_cv = GridSearchCV(estimator= pipe1,
                    param_grid= {
                        'Lg__l1_ratio':[0,0.2,0.4,0.6,0.8,1],
                        'Lg__C':[1000,10000]
                    },cv=5,scoring ='recall')
g_cv.fit(X_train,y_train.values.flatten())





# pipe1.fit(X_train,y_train)
# accuracy_score(y_train,pipe1.predict(X_train))
g_cv.best_estimator_

c:\Users\91972\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:953: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Users\91972\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 942, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "c:\Users\91972\anaconda3\Lib\site-packages\sklearn\metrics\_scorer.py", line 308, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\91972\anaconda3\Lib\site-packages\sklearn\metrics\_scorer.py", line 400, in _score
    y_pred = method_caller(
        estimator,
    ...<2 lines>...
        pos_label=pos_label,
    )
  File "c:\Users\91972\anaconda3\Lib\site-packages\sklearn\metrics\_scorer.py", line 90, in _cached_call
    r

,steps,"[('Std', ...), ('Lg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('Stand', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


g_cv.best_params_

In [118]:
g_cv.best_params_
model = g_cv.best_estimator_
model.fit(X_train,y_train.values.flatten())
y_trained = model.predict(X_train)
from sklearn.metrics import classification_report
print(classification_report(y_train,y_trained))


              precision    recall  f1-score   support

     Absence       0.85      0.87      0.86       119
    Presence       0.84      0.80      0.82        97

    accuracy                           0.84       216
   macro avg       0.84      0.84      0.84       216
weighted avg       0.84      0.84      0.84       216



In [119]:
X_train.columns

Index(['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120',
       'EKG results', 'Max HR', 'Exercise angina', 'ST depression',
       'Slope of ST', 'Number of vessels fluro', 'Thallium'],
      dtype='object')

In [120]:
# Pipe line  , Polynomial Logistic Regression
num_col = ['Age','Chest pain type', 'BP', 'Cholesterol', 'FBS over 120',
       'EKG results', 'Max HR', 'Exercise angina', 'ST depression',
       'Slope of ST', 'Number of vessels fluro', 'Thallium']

pipe_num=Pipeline([('Poly',PolynomialFeatures()),
                   ('std',StandardScaler())])
CT = ColumnTransformer(transformers=[('Num',pipe_num,num_col)],remainder='passthrough')

pipe = Pipeline([('preprocessed',CT),
                 ('Lg',LogisticRegression(max_iter=10000,penalty='elasticnet',solver='saga'))])

# # Using Grid Search CV
from sklearn.model_selection import GridSearchCV

g_cv2 = GridSearchCV(estimator= pipe,
                    param_grid= {
                        'preprocessed__Num__Poly__degree':[2,3],
                        'preprocessed__Num__Poly__include_bias':[True,False],
                        'Lg__l1_ratio':[0,0.2,0.4,0.6,0.8,1],
                        'Lg__C':[1000,10000]
                    },cv=5,scoring ='recall')
g_cv2.fit(X_train,y_train)




pipe
# # pipe1.fit(X_train,y_train)
# # accuracy_score(y_train,pipe1.predict(X_train))
# # g_cv.best_estimator_

c:\Users\91972\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\91972\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:953: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Users\91972\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 942, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "c:\Users\91972\anaconda3\Lib\site-packages\sklearn\metrics\_scorer.py", line 308, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\91972\anaconda3\Lib\site-pack

,steps,"[('preprocessed', ...), ('Lg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('Num', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [121]:
g_cv2.best_params_

{'Lg__C': 1000,
 'Lg__l1_ratio': 0,
 'preprocessed__Num__Poly__degree': 2,
 'preprocessed__Num__Poly__include_bias': True}

In [122]:
model2 = g_cv2.best_estimator_
model2.fit(X_train,y_train.values.flatten())

,steps,"[('preprocessed', ...), ('Lg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('Num', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [123]:
print(classification_report(y_train,model2.predict(X_train)))

              precision    recall  f1-score   support

     Absence       0.94      0.97      0.96       119
    Presence       0.97      0.93      0.95        97

    accuracy                           0.95       216
   macro avg       0.96      0.95      0.95       216
weighted avg       0.95      0.95      0.95       216



In [124]:
print(confusion_matrix(y_test,model2.predict(X_test)))
print((accuracy_score(y_test,model2.predict(X_test))))
print(precision_score(y_test,model2.predict(X_test),average=None))
print(recall_score(y_test,model2.predict(X_test),average=None))

[[29  2]
 [ 4 19]]
0.8888888888888888
[0.87878788 0.9047619 ]
[0.93548387 0.82608696]


In [125]:
from sklearn.metrics import roc_auc_score
print(roc_auc_score(y_test, model2.predict_proba(X_test)[:, 1]))


0.9186535764375877


In [136]:
# Pipe line 3 , Polynomial Logistic Regression with PCA

from sklearn.decomposition import PCA

pipe_num=Pipeline([("pca",PCA()),
                   ('Poly',PolynomialFeatures(include_bias=True)),
                   ('std',StandardScaler())])
CT = ColumnTransformer(transformers=[('Num',pipe_num,num_col)],remainder='passthrough')

pipe = Pipeline([('preprocessed',CT),
                 ('Lg',LogisticRegression(max_iter=10000,penalty='elasticnet',solver='saga'))])

# # Using Grid Search CV
from sklearn.model_selection import GridSearchCV

g_cv3 = GridSearchCV(estimator= pipe,
                    param_grid= {
                        'preprocessed__Num__pca__n_components':range(2,10),
                        'preprocessed__Num__Poly__degree':[2],
                        'Lg__l1_ratio':[0,0.6,1],
                    },cv=5,scoring ='recall')
g_cv3.fit(X_train,y_train)




# # pipe1.fit(X_train,y_train)
# # accuracy_score(y_train,pipe1.predict(X_train))
# # g_cv.best_estimator_

c:\Users\91972\anaconda3\Lib\site-packages\sklearn\utils\validation.py:1406: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\91972\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:953: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Users\91972\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 942, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "c:\Users\91972\anaconda3\Lib\site-packages\sklearn\metrics\_scorer.py", line 308, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\91972\anaconda3\Lib\site-pack

,estimator,Pipeline(step...ver='saga'))])
,param_grid,"{'Lg__l1_ratio': [0, 0.6, ...], 'preprocessed__Num__Poly__degree': [2], 'preprocessed__Num__pca__n_components': range(2, 10)}"
,scoring,'recall'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('Num', ...)]"


In [141]:
model3 =g_cv3.best_estimator_
g_cv3.best_params_

print(confusion_matrix(y_test,model3.predict(X_test)))
print((accuracy_score(y_test,model3.predict(X_test))))
print(precision_score(y_test,model3.predict(X_test),average=None))
print(recall_score(y_test,model3.predict(X_test),average=None))

[[27  4]
 [ 7 16]]
0.7962962962962963
[0.79411765 0.8       ]
[0.87096774 0.69565217]


NameError: name 'hi' is not defined